# **Data-Ingestion:**

## **Load the Environments:**

In [1]:
# Load the Secrects fom .env:

from dotenv import load_dotenv
import os

# load the .env file
load_dotenv()


# Get the secrets from the environment variables:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL")
OPENAI_CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL")


MONGO_URI = os.getenv("MONGO_URI")
MONGO_DB = os.getenv("MONGO_DB")

## **Get the Embeddings Model:**

In [2]:
from openai import OpenAI


# Initialize client
client = OpenAI()


# Specify the embedding model
model = OPENAI_EMBEDDING_MODEL


def get_embedding(text):
    """
        Generate an embedding for the given text using OpenAI's API.
    """
    try:
        response = client.embeddings.create(
            model=model,
            input=[text]
        )
        # print(response.data[0])

        return response.data[0].embedding

    except Exception as e:
        print(f"Error generating embedding: {e}")
        return None



In [3]:
embedding = get_embedding("Hello, how are you?")
print(embedding)
print(f"Embedding length: {len(embedding)}")

[0.0230712890625, -0.050018310546875, -0.01172637939453125, 0.043609619140625, -0.00855255126953125, -0.022857666015625, 0.011077880859375, 0.02117919921875, -0.0192108154296875, -0.07525634765625, -0.0121612548828125, -0.0157012939453125, 0.0016603469848632812, -0.0487060546875, 0.004547119140625, 0.050872802734375, -0.0255584716796875, 0.05755615234375, -0.00931549072265625, 0.03369140625, 0.0650634765625, -0.002864837646484375, 0.00936126708984375, 0.0227508544921875, 0.040985107421875, 0.0128631591796875, 0.00919342041015625, -0.00461578369140625, 0.00994873046875, -0.01294708251953125, 0.07037353515625, -0.032562255859375, -0.003265380859375, 0.0289459228515625, -0.0151519775390625, 0.0137939453125, -0.01226043701171875, 0.014984130859375, -0.0027561187744140625, -0.0711669921875, 0.002513885498046875, -0.0236358642578125, 0.006359100341796875, 0.022552490234375, 0.00731658935546875, 0.0081787109375, 0.0037708282470703125, -0.0260772705078125, 0.0491943359375, 0.01032257080078125,

## **Data Loading & Splitting the Content:**

* Load the Pdf
* Split the Text by using custom Recursive Character Splitter

### **Load Pdf:**

In [4]:
# Load the Pdf:

from IPython.display import display, Markdown
import pdfplumber


def extract_text_from_pdf(pdf_path):
    """
        Extract text from a PDF file using pdfplumber.
    """
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = ""
            for page in pdf.pages:
                text += page.extract_text() + "\n"

        return text
    
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return None
    





In [5]:
content = extract_text_from_pdf(pdf_path="sample.pdf")
# display(Markdown(content))

### **Custom Recursive Character Splitter:**

In [6]:
from typing import Any



class RecursiveCharacterSplitter:
    """
        Pure-Python recursive character text splitter with configurable chunk size, overlap, and separator hierarchy.
    """

    # Default separators: try to split on paragraphs, then lines, then sentences, then words, then characters.
    DEFAULT_SEPARATORS: list[str] = ["\n\n", "\n", ". ", " ", ""]

    def __init__(
        self,
        chunk_size: int = 1000,
        chunk_overlap: int = 200,
        separators: list[str] | None = None,
    ) -> None:
        try:
            if not isinstance(chunk_size, int) or chunk_size <= 0:
                raise ValueError(f"chunk_size must be a positive integer, got: {chunk_size}")
            
            if not isinstance(chunk_overlap, int) or chunk_overlap < 0:
                raise ValueError(f"chunk_overlap must be a non-negative integer, got: {chunk_overlap}")
            
            if chunk_overlap >= chunk_size:
                raise ValueError(
                    f"chunk_overlap ({chunk_overlap}) must be less than chunk_size ({chunk_size})"
                )

            self.chunk_size = chunk_size
            self.chunk_overlap = chunk_overlap
            self.separators = separators if separators is not None else self.DEFAULT_SEPARATORS

        except ValueError:
            raise

        except Exception as exc:
            raise RuntimeError(f"Failed to initialize RecursiveCharacterSplitter: {exc}") from exc


    def _split_text(self, text: str, separators: list[str]) -> list[str]:
        """
            Recursively split ``text`` using ``separators``.

            Tries the first separator; any resulting piece that still exceeds
            ``chunk_size`` is passed back recursively with the remaining
            separators. The empty-string separator acts as a character-level
            fallback that always succeeds.
        """
        try:
            if not text:
                return []

            separator = separators[0]
            remaining = separators[1:]

            if separator == "":
                # Character-level split — hard-chop into chunk_size pieces.
                return [text[i: i + self.chunk_size] for i in range(0, len(text), self.chunk_size)]

            raw_splits = text.split(separator)
            pieces: list[str] = []
            current = ""

            for part in raw_splits:
                candidate = (current + separator + part).lstrip(separator) if current else part
                if len(candidate) <= self.chunk_size:
                    current = candidate
                else:
                    if current:
                        pieces.append(current)
                    # Part alone is too big — recurse with finer separators.
                    if len(part) > self.chunk_size and remaining:
                        pieces.extend(self._split_text(part, remaining))
                    
                    else:
                        current = part

            if current:
                pieces.append(current)

            return [p for p in pieces if p.strip()]

        except RecursionError:
            return [text[i: i + self.chunk_size] for i in range(0, len(text), self.chunk_size)]

        except Exception as exc:
            raise


    def _apply_overlap(self, pieces: list[str]) -> list[str]:
        """
            Stitch an overlap tail from each chunk into the beginning of the next.
        """
        try:
            if len(pieces) <= 1 or self.chunk_overlap == 0:
                return pieces

            result: list[str] = [pieces[0]]
            for i in range(1, len(pieces)):
                tail = result[-1][-self.chunk_overlap:]
                merged = tail + pieces[i]
                result.append(merged)

            return result

        except Exception as exc:
            raise


    def split_text(self, text: str) -> list[str]:
        """
            Split a single string into overlapping chunks.
        """
        try:
            if not text or not text.strip():
                return []

            pieces = self._split_text(text, self.separators)
            chunks = self._apply_overlap(pieces)

            return chunks

        except Exception as exc:
            raise


    def split_documents(
        self,
        documents: list[dict],
        extra_metadata: dict[str, Any] | None = None,
    ) -> list[dict]:
        """
            Split a list of page dicts (output of PDFLoader.load) into chunks.

            Each page dict must have ``"content"`` and ``"metadata"`` keys.
            ``extra_metadata`` is merged into every chunk's metadata, letting
            callers attach document-level tags (e.g. kb_id, doc_title).

            Returns a list of chunk dicts:
                {
                    "content":     str,
                    "chunk_index": int,   # global index across all pages
                    "token_count": int,   # word-based approximation
                    "metadata":    dict,  # page meta + extra_metadata + chunk_index
                }
        """
        try:
            if not documents:
                return []

            extra = extra_metadata or {}
            chunks: list[dict] = []
            global_index = 0
            skipped_pages = 0

        
            for page_num, doc in enumerate(documents, start=1):
                try:
                    page_content: str = doc.get("content", "").strip()
                    page_meta: dict = doc.get("metadata", {})

                    if not page_content:
                        skipped_pages += 1
                        continue

                    pieces = self.split_text(page_content)

                    for piece in pieces:
                        piece = piece.strip()
                        if not piece:
                            continue

                        chunk_meta = {**page_meta, **extra, "chunk_index": global_index}
                        chunks.append({
                            "content": piece,
                            "chunk_index": global_index,
                            "token_count": len(piece.split()),
                            "metadata": chunk_meta,
                        })
                        global_index += 1

                except Exception as page_exc:
                    skipped_pages += 1
                    

           
            return chunks

        except Exception as exc:
            raise

In [7]:
recursive_splitter = RecursiveCharacterSplitter(chunk_size=1000, chunk_overlap=200)


chunks1 = recursive_splitter.split_documents(documents=[{"content": content}], extra_metadata={"source": "sample.pdf", "type":"pdf", "name":"dibyendu"})

chunks2 = recursive_splitter.split_documents(documents=[{"content": content}], extra_metadata={"source": "sample.pdf", "type":"docs", "name":"arko"})

In [8]:
chunks2

[{'content': '1 Introduction\nRecurrentneuralnetworks,longshort-termmemory[13]andgatedrecurrent[7]neuralnetworks\ninparticular,havebeenfirmlyestablishedasstateoftheartapproachesinsequencemodelingand\ntransductionproblemssuchaslanguagemodelingandmachinetranslation[35,2,5]. Numerous\neffortshavesincecontinuedtopushtheboundariesofrecurrentlanguagemodelsandencoder-decoder\narchitectures[38,24,15].\nRecurrentmodelstypicallyfactorcomputationalongthesymbolpositionsoftheinputandoutput\nsequences. Aligningthepositionstostepsincomputationtime,theygenerateasequenceofhidden\nstatesh ,asafunctionoftheprevioushiddenstateh andtheinputforpositiont. Thisinherently\nt t−1\nsequentialnatureprecludesparallelizationwithintrainingexamples,whichbecomescriticalatlonger\nsequencelengths,asmemoryconstraintslimitbatchingacrossexamples. Recentworkhasachieved\nsignificantimprovementsincomputationalefficiencythroughfactorizationtricks[21]andconditional\ncomputation[32],whilealsoimprovingmodelperformanceincaseofthel

In [9]:
len(chunks1)

6

In [10]:
display(Markdown(chunks1[0]["content"]))

1 Introduction
Recurrentneuralnetworks,longshort-termmemory[13]andgatedrecurrent[7]neuralnetworks
inparticular,havebeenfirmlyestablishedasstateoftheartapproachesinsequencemodelingand
transductionproblemssuchaslanguagemodelingandmachinetranslation[35,2,5]. Numerous
effortshavesincecontinuedtopushtheboundariesofrecurrentlanguagemodelsandencoder-decoder
architectures[38,24,15].
Recurrentmodelstypicallyfactorcomputationalongthesymbolpositionsoftheinputandoutput
sequences. Aligningthepositionstostepsincomputationtime,theygenerateasequenceofhidden
statesh ,asafunctionoftheprevioushiddenstateh andtheinputforpositiont. Thisinherently
t t−1
sequentialnatureprecludesparallelizationwithintrainingexamples,whichbecomescriticalatlonger
sequencelengths,asmemoryconstraintslimitbatchingacrossexamples. Recentworkhasachieved
significantimprovementsincomputationalefficiencythroughfactorizationtricks[21]andconditional
computation[32],whilealsoimprovingmodelperformanceincaseofthelatter. Thefundamental

In [11]:
display(Markdown(chunks1[1]["content"]))

. Recentworkhasachieved
significantimprovementsincomputationalefficiencythroughfactorizationtricks[21]andconditional
computation[32],whilealsoimprovingmodelperformanceincaseofthelatter. Thefundamentalconstraintofsequentialcomputation,however,remains.
Attentionmechanismshavebecomeanintegralpartofcompellingsequencemodelingandtransduc-
tionmodelsinvarioustasks,allowingmodelingofdependencieswithoutregardtotheirdistancein
theinputoroutputsequences[2,19]. Inallbutafewcases[27],however,suchattentionmechanisms
areusedinconjunctionwitharecurrentnetwork.
InthisworkweproposetheTransformer,amodelarchitectureeschewingrecurrenceandinstead
relyingentirelyonanattentionmechanismtodrawglobaldependenciesbetweeninputandoutput.
TheTransformerallowsforsignificantlymoreparallelizationandcanreachanewstateoftheartin
translationqualityafterbeingtrainedforaslittleastwelvehoursoneightP100GPUs.
2 Background
ThegoalofreducingsequentialcomputationalsoformsthefoundationoftheExtendedNeuralGPU
[16],ByteNet[18]andConvS2S[9],allofwhichuseconvolutionalneuralnetworksasbasicbuilding
block,computinghiddenrepresentationsinparallelforallinputandoutputpositions.Inthesemodels,

## **Vector Store:**

* Generate the Embeddings
* Store the Embeddings

In [27]:
from pymongo.operations import SearchIndexModel
from pymongo import MongoClient
from openai import OpenAI
import time


class VectorStore:
    def __init__(
        self,
        mongo_uri: str,
        mongo_db: str,
        collection_name: str,
        openai_client: OpenAI,
        embedding_model: str,
        embedding_dimensions: int = 1536,
        vector_index_name: str = "vector_index",
        search_index_name: str = "search_index",
    ):
        try:
            self.client = MongoClient(mongo_uri)
            self.db = self.client[mongo_db]
            self.collection = self.db[collection_name]
            self.openai_client = openai_client
            self.embedding_model = embedding_model
            self.embedding_dimensions = embedding_dimensions
            self.vector_index_name = vector_index_name
            self.search_index_name = search_index_name

        except Exception as exc:
            raise RuntimeError(f"Failed to initialize VectorStore: {exc}") from exc

    def generate_embedding(self, text: str) -> list[float]:
        try:
            response = self.openai_client.embeddings.create(
                model=self.embedding_model,
                input=[text],
            )
            return response.data[0].embedding

        except Exception as exc:
            raise RuntimeError(f"Failed to generate embedding: {exc}") from exc

    
    def create_vector_index(
        self,
        similarity: str = "cosine",
        extra_filter_fields: list[str] | None = None,
    ) -> None:
        """
            Create an Atlas Vector Search index (type='vectorSearch') on 'embedding'.

            similarity: "cosine" | "euclidean" | "dotProduct"
            extra_filter_fields: additional metadata dot-paths to register as
                                filter fields beyond the defaults.
            Requires Atlas M10+. Builds asynchronously — wait for READY status.
        """
        try:
            base_filters = [
                "metadata.source",
                "metadata.ingested_at",
                "metadata.name",
                "metadata.type",
                "chunk_index",
            ]
            all_filters = base_filters + (extra_filter_fields or [])

            fields = [
                {
                    "type": "vector",
                    "path": "embedding",
                    "numDimensions": self.embedding_dimensions,
                    "similarity": similarity,
                }
            ] + [{"type": "filter", "path": p} for p in all_filters]

            index_model = SearchIndexModel(
                definition={"fields": fields},
                name=self.vector_index_name,
                type="vectorSearch",
            )
            self.collection.create_search_indexes([index_model])
            print(f"Vector search index '{self.vector_index_name}' creation initiated.")
            print(f"Filter fields indexed: {all_filters}")

        except Exception as exc:
            raise RuntimeError(f"Failed to create vector index: {exc}") from exc


    def create_search_index(self) -> None:
        """
            Create an Atlas Search index (type='search') on the 'content' field.

            This is a full Atlas Search index — NOT a basic MongoDB $text index.
            It powers the $search stage inside hybrid_search().
            Requires Atlas M10+. Builds asynchronously — wait for READY status.
        """
        try:
            index_model = SearchIndexModel(
                definition={
                    "mappings": {
                        "dynamic": False,
                        "fields": {
                            "content": [{"type": "string"}],
                        },
                    }
                },
                name=self.search_index_name,
                type="search",
            )
            self.collection.create_search_indexes([index_model])
            print(f"Atlas Search index '{self.search_index_name}' creation initiated.")

        except Exception as exc:
            raise RuntimeError(f"Failed to create search index: {exc}") from exc

    
    def update_vector_index(
        self,
        similarity: str = "cosine",
        filter_fields: list[str] | None = None,
    ) -> None:
        """
            Update the existing vector index with a new filter field list.
            Replaces the entire filter definition — include all fields you need.
            Atlas rebuilds asynchronously — wait for READY before querying.
        """
        try:
            filter_paths = filter_fields or [
                "metadata.source",
                "metadata.ingested_at",
                "metadata.name",
                "metadata.type",
                "chunk_index",
            ]
            fields = [
                {
                    "type": "vector",
                    "path": "embedding",
                    "numDimensions": self.embedding_dimensions,
                    "similarity": similarity,
                }
            ] + [{"type": "filter", "path": p} for p in filter_paths]

            self.collection.update_search_index(
                name=self.vector_index_name,
                definition={"fields": fields},
            )
            print(f"Vector index '{self.vector_index_name}' update initiated.")
            print(f"New filter fields: {filter_paths}")

        except Exception as exc:
            raise RuntimeError(f"Failed to update vector index: {exc}") from exc

    
    def add_chunks(self, chunks: list[dict]) -> list:
        """
            Embed each chunk's 'content', attach the vector as 'embedding',
            stamp 'metadata.ingested_at', and bulk-insert into MongoDB.

            Expected chunk shape (from RecursiveCharacterSplitter.split_documents):
                {"content": str, "chunk_index": int, "token_count": int, "metadata": dict}
        """
        try:
            if not chunks:
                return []

            timestamp = time.time()
            docs = []
            for chunk in chunks:
                embedding = self.generate_embedding(chunk["content"])
                doc = {**chunk, "embedding": embedding}
                doc["metadata"]["ingested_at"] = timestamp
                docs.append(doc)

            result = self.collection.insert_many(docs)
            return result.inserted_ids

        except Exception as exc:
            raise RuntimeError(f"Failed to add chunks: {exc}") from exc

    

    def similarity_search(
        self,
        query: str,
        k: int = 5,
        num_candidates: int = 100,
        filter: dict | None = None,
        min_score: float = 0.0,
    ) -> list[dict]:
        """
        Pure vector similarity search via Atlas $vectorSearch (ANN).
        Each result includes a 'score' field (0-1, higher = more similar).

        filter: MQL pre-filter on indexed fields, e.g. {"metadata.name": "arko"}
        min_score: drop results below this threshold.
        """
        try:
            query_embedding = self.generate_embedding(query)

            vector_search: dict = {
                "index": self.vector_index_name,
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": max(num_candidates, k * 10),
                "limit": k,
            }
            if filter:
                vector_search["filter"] = filter

            pipeline = [
                {"$vectorSearch": vector_search},
                {
                    "$project": {
                        "embedding": 0,
                        "score": {"$meta": "vectorSearchScore"},
                    }
                },
            ]
            if min_score > 0.0:
                pipeline.append({"$match": {"score": {"$gte": min_score}}})

            return list(self.collection.aggregate(pipeline))

        except Exception as exc:
            raise RuntimeError(f"Similarity search failed: {exc}") from exc

    def hybrid_search(
        self,
        query: str,
        k: int = 5,
        num_candidates: int = 100,
        vector_weight: float = 0.7,
        fulltext_weight: float = 0.3,
        filter: dict | None = None,
    ) -> list[dict]:
        """
            Hybrid search using MongoDB's native $rankFusion pipeline stage.

            Runs two sub-pipelines inside a single aggregation and fuses their
            ranked results using Reciprocal Rank Fusion with configurable weights:

            vector   sub-pipeline: $vectorSearch on 'embedding'
            fulltext sub-pipeline: $search (Atlas Search) on 'content'

            Requires both indexes to be in READY state:
            - vector_index  → created with create_vector_index()
            - search_index  → created with create_search_index()   ← Atlas Search

            filter: MQL filter applied to $vectorSearch pre-filter AND as a
                    post-$rankFusion $match, so both sub-pipelines' results
                    are correctly scoped.

            Note: $rankFusion is an Atlas preview feature (Atlas M10+ required).
        """
        try:
            query_embedding = self.generate_embedding(query)

            vector_stage: dict = {
                "index": self.vector_index_name,
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": max(num_candidates, k * 10),
                "limit": k,
            }
            if filter:
                vector_stage["filter"] = filter

            pipeline = [
                {
                    "$rankFusion": {
                        "input": {
                            "pipelines": {
                                "vector": [
                                    {"$vectorSearch": vector_stage}
                                ],
                                "fulltext": [
                                    {
                                        "$search": {
                                            "index": self.search_index_name,
                                            "text": {
                                                "query": query,
                                                "path": "content",
                                            },
                                        }
                                    },
                                    {"$limit": k},
                                ],
                            }
                        },
                        "combination": {
                            "weights": {
                                "vector": vector_weight,
                                "fulltext": fulltext_weight,
                            }
                        },
                    }
                },
                # Post-fusion filter — enforces filter on fulltext results too,
                # since $search inside $rankFusion cannot accept a pre-filter.
                *([ {"$match": filter} ] if filter else []),
                {"$limit": k},
                {"$project": {"embedding": 0}},
            ]

            return list(self.collection.aggregate(pipeline))

        except Exception as exc:
            raise RuntimeError(f"Hybrid search failed: {exc}") from exc


In [28]:
# VectorStore initialization
vector_store = VectorStore(
    mongo_uri=MONGO_URI,
    mongo_db=MONGO_DB,
    collection_name="documents",
    openai_client=OpenAI(api_key=OPENAI_API_KEY),
    embedding_model=OPENAI_EMBEDDING_MODEL,
    embedding_dimensions=1536,
    vector_index_name="vector_index",
    search_index_name="search_index",   # Atlas Search index for hybrid_search()
)

# # One-time setup — run once per collection, then comment out
# vector_store.create_vector_index(similarity="cosine")  # Atlas vectorSearch index
# vector_store.create_search_index()                     # Atlas Search index (for $rankFusion)

In [14]:
# Add the chunks to the VectorStore (generates embeddings and inserts into MongoDB)
inserted_ids = vector_store.add_chunks(chunks1)
print(f"Inserted {len(inserted_ids)} chunks into VectorStore.")


inserted_ids = vector_store.add_chunks(chunks2)
print(f"Inserted {len(inserted_ids)} chunks into VectorStore.")

Inserted 6 chunks into VectorStore.
Inserted 6 chunks into VectorStore.


In [15]:
# Do the Similarity Search:
similar_docs = vector_store.similarity_search(query="What is the neural networks?", k=3, num_candidates=100)
similar_docs

[{'_id': ObjectId('6a09b870afa0d660ca941aa9'),
  'content': 'noftheExtendedNeuralGPU\n[16],ByteNet[18]andConvS2S[9],allofwhichuseconvolutionalneuralnetworksasbasicbuilding\nblock,computinghiddenrepresentationsinparallelforallinputandoutputpositions.Inthesemodels,thenumberofoperationsrequiredtorelatesignalsfromtwoarbitraryinputoroutputpositionsgrows\ninthedistancebetweenpositions,linearlyforConvS2SandlogarithmicallyforByteNet. Thismakes\nit more difficult to learn dependencies between distant positions [12]. In the Transformer this is\nreducedtoaconstantnumberofoperations, albeitatthecostofreducedeffectiveresolutiondue\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\ndescribedinsection3.2.\nSelf-attention,sometimescalledintra-attentionisanattentionmechanismrelatingdifferentpositions\nofasinglesequenceinordertocomputearepresentationofthesequence. Self-attentionhasbeen\nusedsuccessfullyinavarietyoftasksincludingreadingcomprehension,abstract

In [18]:
# Similarity Search with a filter:

vector_store.similarity_search(
    query="What is the neural networks?",
    k=3,
    filter={"metadata.name": "dibyendu"}, 
)


[{'_id': ObjectId('6a09b870afa0d660ca941aa9'),
  'content': 'noftheExtendedNeuralGPU\n[16],ByteNet[18]andConvS2S[9],allofwhichuseconvolutionalneuralnetworksasbasicbuilding\nblock,computinghiddenrepresentationsinparallelforallinputandoutputpositions.Inthesemodels,thenumberofoperationsrequiredtorelatesignalsfromtwoarbitraryinputoroutputpositionsgrows\ninthedistancebetweenpositions,linearlyforConvS2SandlogarithmicallyforByteNet. Thismakes\nit more difficult to learn dependencies between distant positions [12]. In the Transformer this is\nreducedtoaconstantnumberofoperations, albeitatthecostofreducedeffectiveresolutiondue\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\ndescribedinsection3.2.\nSelf-attention,sometimescalledintra-attentionisanattentionmechanismrelatingdifferentpositions\nofasinglesequenceinordertocomputearepresentationofthesequence. Self-attentionhasbeen\nusedsuccessfullyinavarietyoftasksincludingreadingcomprehension,abstract

In [31]:
# Hybrid Search:
hybrid_results = vector_store.hybrid_search(
    query="What is the neural networks?",
    k=3,
    vector_weight=0.7,
    fulltext_weight=0.3,
    filter={"metadata.name": "dibyendu"},  
)
hybrid_results

[{'_id': ObjectId('6a09b870afa0d660ca941aa9'),
  'content': 'noftheExtendedNeuralGPU\n[16],ByteNet[18]andConvS2S[9],allofwhichuseconvolutionalneuralnetworksasbasicbuilding\nblock,computinghiddenrepresentationsinparallelforallinputandoutputpositions.Inthesemodels,thenumberofoperationsrequiredtorelatesignalsfromtwoarbitraryinputoroutputpositionsgrows\ninthedistancebetweenpositions,linearlyforConvS2SandlogarithmicallyforByteNet. Thismakes\nit more difficult to learn dependencies between distant positions [12]. In the Transformer this is\nreducedtoaconstantnumberofoperations, albeitatthecostofreducedeffectiveresolutiondue\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\ndescribedinsection3.2.\nSelf-attention,sometimescalledintra-attentionisanattentionmechanismrelatingdifferentpositions\nofasinglesequenceinordertocomputearepresentationofthesequence. Self-attentionhasbeen\nusedsuccessfullyinavarietyoftasksincludingreadingcomprehension,abstract